# Integrated Intelligent Systems: Tutorial 1 - Foundations (3 Pts)

This notebook introduces the fundamental components, challenges, and definitions, central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: The Unified Theoretical Exercise (E1–E6) (1.5 Pt)

**Title:** From Natural Language Request to action execution: The Eva Coffee Scenario

![Alternative Text](coffee-task-photo.png)

### Scenario Background
Eva requests: "Bring me my coffee." Her usual blue mug is in the dishwasher; a white mug is available on the counter. The fill level can vary, and Eva may shift position during execution.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)
Students work through this unified exercise in the VRB or on paper, structured across the six developmental steps of Lecture 1:

1. **E1 · Opening Judgment (Data vs. Competence):**
   - If an end-to-end vision-language-action model or a demonstration video shows a robot successfully serving coffee 100 times in a row, does that prove the robot "can do the task" across any home environment? State your reasoning and rate your confidence from 1 to 5.
     

2. **E2 · Changed Case Prediction (Physical Bounds):**
   - A half-full white mug was carried successfully along the route. Suppose the mug is now nearly full. Predict the physical consequence during a turn, and identify the untested assumption in the original run.
     

3. **E3 · The Commitment Decision (Warrant Gate):**
   - The blue mug is unavailable and white-mug acceptance is unconfirmed. Choose whether the robot should ask, observe, or proceed. State the single item of evidence that justifies your choice.
     

4. **E4 · The Five Questions of Understanding (T1–T5):**
   - Formulate one distinct question for each dimension—Explain ($\text{T1}$), Predict ($\text{T2}$), Design ($\text{T3}$), Bound ($\text{T4}$), and Check ($\text{T5}$)—concerning the coffee delivery episode. At least one question must address a changed condition that a video alone cannot answer.
     

5. **E5 · System Decomposition (Mapping to Architecture):**
   - List the core functional modules required to handle Eva's request (interpreting, perceiving, proposing, predicting, warranting, acting, and recording) and map them onto the Integration Diagram.
     

6. **E6 · Transfer & Exit Ticket (Justified Adaptation):**
   - Suppose Eva stands up while the robot is carrying the coffee. Revise the action, predict the potential failure mechanism, and state the exact evidence criterion required before the robot commits to the revised path.

## Answers

**E1:** I don't think that the robot will be able to perfectly do the task across any home environment. The real world is fuzzy, with lots of variability and unexpected events. Slight imperfections in floor tiling, different lighting conditions, interactions with humans and other agents, obstacles, sudden position changes for people and/or things, and vague or unclear environments or furniture can all interfere with the robot and the virtual vision-language-action model didn't have such real world experience. Furthermore, the patterns that the vision-language-action model will not translate smoothly with the real world. I am 5/5 sure that the robot will need to interact with the real world and be tested there. 

**E2:** The robot might have turned with a high velocity and caused a high jerk that was fine with a half-full mug, but might not be fine with a nearly full mug. The coffee might slosh from the fast movements of the robot and spill out.

**E3:** The robot should ask for clarification from Eva. It should ground the request. Eva only asked for coffee, but she is used to her blue mug and might ask the robot to look for it, or, if the blue mug needs to be washed, to wash it and then bring it. Eva could also be fine with a white mug, but the robot doesn't know anything for sure at this point. Clarifying with Eva will remove any doubts and cost nothing in terms of resources. There is also no indication from Eva's request that she is in a rush.

**E4**: ***Wet mug vs Dry mug scenario***

**Explain**: Did the mug not slip because the robot applied correct gripping force, or because the mug's surface was dry?
        
**Predict**: Will a different mug that is wet slip if the robot uses the same gripping strategy? The robot must predict what will happen if a mug with               same material will slip if wet or not with same grip, so that it can learn to adjust the grip.
        
**Design**:  Would a different gripper or gripping strategy improve the outcome? Coating the gripper with rubber or wool might help in case the mug's                surface is slippery. 

**Bound**: Which mug materials, surface conditions, weights, and grippers are covered by the robot's experience?

**Check**: Is there evidence that the mug was delivered or did it slip and break? The robot's perception should verify the delivery wthout accidents.              Also, is Eva fine with receiving a wet mug? Or does she expect her mug to be dry? How safe is it to handle someone a wet coffee mug?


**E5**: The robot should ***interepret*** Eva's request and ground the request to learn what drink she wants, which mug and how quick she wants it delivered.The robot should then ***perceive*** the state of the environment, and pay attention to what matters to the task at hand; figuring out where the coffee mug is and the coffee packet and so on. The robot ***proposes*** routes and actions. The ***digital twin*** should then simulate these actions. For example, before opening Eva's dishwasher, the robot  might use a digital twin to predict the door's motion, thereach to its handle, and whether the open door will obstruct her walking frame. Simulation does not guarantee what the physical world will do. A useful twin produces predictions that observations can test; disagreement should update the model or interrupt the action. The robot then sits out to ***decide and warrant*** the action; by executing, monitoring, and revising the strategy accoring to changes perceived and safety requirements. For example, if the robot picked a route velocity that is causes a spill, it should changed the velocity and it's strategy. Therefor, the robot must ***regulate it's actions*** constantly and not just stubbornly pick an action and not respond to the dynamic environment. Finally the robot should keep a ***record*** of its actions and justifications for the specific choices it made. 


**E6**: The robot should detect Eva's change in position. The robot should stop and reassess the picked route. It should select an alternative route that maintains a safe distance from Eva. Possible failures include a collision with Eva, the robot changes or stops too quickly causing the coffee to spill, or running into an obstacle on the alternative route. The robot should predict these failures and mitigate them while constantly regulating its actions.The robot must not assume that the alternative path is safe simply because it appears clear. It must obtain sufficient evidence that following the new route will not exceed the safety velocity and acceleration for the liquid not to spill, Eva's movement and trajectory constantly being calculated to keep a safe distance and not collide with her and it must use its sensors to make sure the new route remains obstacle free. 
        

## Part 2: Interactive Virtual Lab & Python Tutorial (1.5 Pt)

**Title:** Implementing the Embodied Cognitive Architecture

### Scenario Background
Eva requests: "Bring me my coffee."

---

In [ ]:
#!/usr/bin/env python3
"""Embodied Cognitive Architecture
"""

from dataclasses import dataclass, field
import math
from typing import Dict, Any, Tuple, Optional, List
import random
import time

@dataclass(frozen=True)
class GoalSpecification:
    """Formalized, concrete goal specification derived via semantic interpretation."""
    target_object: str
    destination: str
    target_volume_ml: float
    max_allowable_force_n: float
    max_slosh_wave_mm: float
    max_nav_velocity_mps: float
    time_frame_seconds: float

@dataclass
class ParameterPacket:
    """System 1 (The Proposer): Flexible sampler proposing execution parameters."""
    grasp_pose: Tuple[float, float, float]
    max_velocity_mps: float
    tilt_rate_radps: float
    source_generator: str = "stochastic_sampler"

class Proposer:
    """Generates randomized ParameterPackets for robot simulation."""
    def __init__(self, min_velocity: float = 0.1, max_velocity: float = 1.2, min_tilt: float = 0.0, max_tilt: float = 0.5, pose_center: Tuple[float, float, float] = (0.40, 0.10, 0.30), pose_std: float = 0.05):
        
        self.pose_center = pose_center
        self.pose_std = pose_std
        
        # Define limits for the robot speed and tilt
        self.min_velocity = min_velocity
        self.max_velocity = max_velocity
        
        self.min_tilt = min_tilt
        self.max_tilt = max_tilt

    def propose(self) -> ParameterPacket:
        """Generates and returns a randomly initialized ParameterPacket."""
        # Gaussian for Grasp Pose
        x = random.gauss(self.pose_center[0], self.pose_std)
        y = random.gauss(self.pose_center[1], self.pose_std)
        z = random.gauss(self.pose_center[2], self.pose_std)
        
        # Uniform for velocity and tilt rate
        max_vel = random.uniform(max(0, self.min_velocity), self.max_velocity)
        tilt_rate = random.uniform(self.min_tilt, self.max_tilt)
        
        return ParameterPacket(
            grasp_pose=(x, y, z),
            max_velocity_mps=max_vel,
            tilt_rate_radps=tilt_rate,
            source_generator="gaussian_uniform_hybrid"
        )

@dataclass
class Observation:
    """Raw, noisy sensor measurements at time t (distinct from internal belief)."""
    timestamp: float
    measured_temperature_c: float
    measured_friction: float
    eva_position: str
    room_busyness: str

@dataclass
class BeliefState:
    """The robot's current internal belief state (updated via perception)."""
    pouch_present: bool = True
    pouch_material: str = "flexible_plastic"
    fluid_viscosity_pas: float = 1.2
    vial_ready: bool = True
    camera_active: bool = True
    fallback_path_exists: bool = True
    audit_logger_enabled: bool = True
    # Real-time environmental parameters maintained internally
    ambient_temperature_c: float = 21.5
    surface_friction_coefficient: float = 0.45
    eva_position: str = "living_room"
    trajectory_distance: float = 1.5  # meters
    battery_capacity_wh: float = 100.0
    low_battery: bool = False
    room_busyness: str = "medium"  # 'low', 'medium', 'high'
    preferred_cup: str = "blue_ceramic_mug"


class InterpretationSystem:
    """Uses a semantic knowledge base to map vague instructions and context 
    into precise physical goal parameters (GoalSpecification).
    """
    def __init__(self, knowledge_base: Dict[str, Any]):
        self.kb = knowledge_base

    def interpret(self, raw_utterance: str, belief: BeliefState) -> GoalSpecification:
        print(f"\n[INTERPRETATION] Analyzing instruction: '{raw_utterance}' in context of Eva (Position: {belief.eva_position}, Room Busyness: {belief.room_busyness})")
        
        cup_type = belief.preferred_cup
        cup_properties = self.kb["cups"].get(cup_type, {"max_force": 5.0, "max_wave": 2.0})
        
        busyness_factor = self.kb["room_busyness_limits"].get(belief.room_busyness, 0.5)
        max_nav_vel = 1.2 * busyness_factor
        
        is_urgent = "schnell" in raw_utterance.lower() or "fast" in raw_utterance.lower()
        time_frame = 30.0 if is_urgent else 60.0
        target_volume = 15.0 if not is_urgent else 12.0
        
        goal = GoalSpecification(
            target_object=cup_type,
            destination=belief.eva_position,
            target_volume_ml=target_volume,
            max_allowable_force_n=cup_properties["max_force"],
            max_slosh_wave_mm=cup_properties["max_wave"],
            max_nav_velocity_mps=max_nav_vel,
            time_frame_seconds=time_frame
        )
        
        print(f" -> Derived Goal: Object={goal.target_object}, MaxForce={goal.max_allowable_force_n}N, MaxNavVel={goal.max_nav_velocity_mps}m/s, TimeFrame={goal.time_frame_seconds}s")
        return goal


class SensingSystem:
    """Simulates real-time noisy hardware sensors emitting raw observations."""
    def __init__(self):
        pass

    def sense(self, true_ambient_temp: float, true_friction: float, eva_pos: str, busyness: str) -> Observation:
        noisy_temp = true_ambient_temp + random.uniform(-0.5, 0.5)
        noisy_friction = true_friction + random.uniform(-0.05, 0.05)
        
        return Observation(
            timestamp=time.time(),
            measured_temperature_c=round(noisy_temp, 2),
            measured_friction=round(noisy_friction, 2),
            eva_position=eva_pos,
            room_busyness=busyness
        )


class PerceptionSystem:
    """Estimator module that updates the internal BeliefState using raw Observations."""
    def __init__(self, belief_state: BeliefState):
        self.belief_state = belief_state

    def update(self, observation: Observation) -> None:
        self.belief_state.ambient_temperature_c = observation.measured_temperature_c
        self.belief_state.surface_friction_coefficient = observation.measured_friction
        self.belief_state.eva_position = observation.eva_position
        self.belief_state.room_busyness = observation.room_busyness


class SemanticDigitalTwin:
    """System 2 (Semantic Digital Twin): Imperfect models with sim-to-real epistemic gaps."""
    def __init__(self, belief_state: BeliefState):
        self.belief = belief_state

    def predict_properties(self, params: ParameterPacket, goal: GoalSpecification) -> Dict[str, float]:
        stiffness_coefficient = 12.5 * (1.0 + self.belief.surface_friction_coefficient * 0.1)


        # Contact depth estimation. the z axis is assumed to be the depth
        nominal_surface_z = 0.30
        penetration_z = max(0.0, nominal_surface_z - params.grasp_pose[2])
                
        emergent_contact_depth = penetration_z * (1.0 + params.max_velocity_mps * 0.2)
                
        # Apply the exponential stiffness as defined in the task
        if emergent_contact_depth > 0.02:
            stiffness_coefficient *= (1.8 ** (emergent_contact_depth / 0.02))
        predicted_force = stiffness_coefficient * (params.max_velocity_mps ** 1.5)
        
        effective_viscosity = self.belief.fluid_viscosity_pas * (1.0 - (self.belief.ambient_temperature_c - 20.0) * 0.01)


        #Arrhenius like implementation of effective viscosity based on temperature.
        '''
        T_ref = 20.0 + 273.15 
        T_ambient = self.belief.ambient_temperature_c + 273.15
        
       
        mu_ref = self.belief.fluid_viscosity_pas
 
        B = 3500.0 
        effective_viscosity = mu_ref * math.exp(B * ((1.0 / T_ambient) - (1.0 / T_ref)))
        '''

        
        radius = 0.05
        wave_height_m = (params.tilt_rate_radps * radius) / effective_viscosity
        predicted_wave = wave_height_m * 1000.0

        

        # Injecting minor epistemic simulation error (Sim-to-Real gap)
        epistemic_force_error = random.uniform(-0.2, 0.3)
        epistemic_wave_error = random.uniform(-0.1, 0.2)

        return {
            "predicted_force_n": round(max(0.0, predicted_force + epistemic_force_error), 2),
            "predicted_wave_mm": round(max(0.0, predicted_wave + epistemic_wave_error), 2),
            "predicted_nav_velocity": min(params.max_velocity_mps, goal.max_nav_velocity_mps),
            "emergent_contact_depth_m": round(emergent_contact_depth, 2)
        }


class WarrantGate:
    """Validates predicted properties against the interpreted GoalSpecification."""
    def __init__(self, sdt: SemanticDigitalTwin):
        self.sdt = sdt
        self.energy_per_meter_wh = 0.5
        
    def validate(self, params: ParameterPacket, goal: GoalSpecification, belief: BeliefState) -> Dict[str, Any]:
        predictions = self.sdt.predict_properties(params, goal)
        estimated_energy_wh = belief.trajectory_distance * self.energy_per_meter_wh
        estimated_drain_percent = (
            estimated_energy_wh / belief.battery_capacity_wh
        ) * 100
        checks = {
            "grounded": belief.vial_ready and belief.pouch_present,
            "feasible": params.max_velocity_mps > 0.0 and params.tilt_rate_radps > 0.0,
            "safe": (predictions["predicted_force_n"] <= goal.max_allowable_force_n) and 
                    (predictions["predicted_wave_mm"] <= goal.max_slosh_wave_mm) and
                    (predictions["predicted_nav_velocity"] <= goal.max_nav_velocity_mps),
            "monitorable": belief.camera_active,
            "repairable": belief.fallback_path_exists,
            "recordable": belief.audit_logger_enabled,
            "energetically_viable": estimated_drain_percent < 15.0
        }
        
        all_passed = all(checks.values())
        failed_criterion = next((name for name, passed in checks.items() if not passed), None)
        
        return {
            "passed": all_passed,
            "failed_criterion": failed_criterion,
            "predictions": predictions,
            "warrant_checks": checks
        }


class ExecutionEngine:
    """Manages physical execution and constructs the epistemic audit trail."""
    def __init__(self):
        self.audit_trail: List[Dict[str, Any]] = []

    def execute_action(self, params: ParameterPacket, goal: GoalSpecification, warrant_result: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n[ACTUATOR] Executing motion in physical world: v_max={params.max_velocity_mps} m/s")
        time.sleep(0.1)
        
        # Real-world divergence check
        predictions = warrant_result["predictions"]
        actual_force = predictions["predicted_force_n"] + random.uniform(-0.3, 0.6)
        actual_wave = predictions["predicted_wave_mm"] + random.uniform(-0.2, 0.5)
        
        spill_occurred = (actual_force > goal.max_allowable_force_n) or (actual_wave > goal.max_slosh_wave_mm)
        status = "FAILURE" if spill_occurred else "SUCCESS"
        
        execution_outcome = {
            "actual_force_n": round(actual_force, 2),
            "actual_wave_mm": round(actual_wave, 2),
            "actual_volume_transferred_ml": goal.target_volume_ml if not spill_occurred else goal.target_volume_ml - 3.0,
            "spill_occurred": spill_occurred,
            "status": status
        }
        return execution_outcome

    def record_trace(self, asked_utterance: str, goal: GoalSpecification, belief: BeliefState, 
                     proposal: ParameterPacket, warrant_result: Dict[str, Any], 
                     observation: Observation, committed: bool, execution_outcome: Optional[Dict[str, Any]] = None) -> None:
        record = {
            "timestamp": time.time(),
            "asked": {
                "raw_utterance": asked_utterance,
                "interpreter_output": {
                    "target": goal.target_object,
                    "max_force_limit_n": goal.max_allowable_force_n,
                    "max_nav_velocity": goal.max_nav_velocity_mps
                }
            },
            "belief": {
                "eva_position": belief.eva_position,
                "room_busyness": belief.room_busyness,
                "ambient_temp_c": belief.ambient_temperature_c,
                "surface_friction": belief.surface_friction_coefficient
            },
            "observed": {
                "timestamp": observation.timestamp,
                "measured_temp_c": observation.measured_temperature_c,
                "measured_friction": observation.measured_friction,
                "eva_position": observation.eva_position,
                "room_busyness": observation.room_busyness
            },
            "committed": {
                "action_approved": committed,
                "proposed_parameters": {
                    "max_velocity_mps": proposal.max_velocity_mps,
                    "tilt_rate_radps": proposal.tilt_rate_radps
                },
                "warrant_verdict": warrant_result
            },
            "execution_outcome": execution_outcome
        }
        self.audit_trail.append(record)


# --- Execution Demonstration ---
if __name__ == "__main__":
    knowledge_base = {
        "cups": {
            "blue_ceramic_mug": {"max_force": 4.5, "max_wave": 1.5},
            "plastic_cup": {"max_force": 7.0, "max_wave": 3.0}
        },
        "room_busyness_limits": {
            "low": 1.0,
            "medium": 0.6,
            "high": 0.3
        }
    }

    # 1. Initialize systems
    belief_state = BeliefState(room_busyness="medium", preferred_cup="blue_ceramic_mug")
    sensing = SensingSystem()
    perception = PerceptionSystem(belief_state)
    interpreter = InterpretationSystem(knowledge_base)
    sdt = SemanticDigitalTwin(belief_state)
    warrant_gate = WarrantGate(sdt)
    executive = ExecutionEngine()
    
    print("=== STEP 1: Sensing & Perception Update ===")
    raw_observation = sensing.sense(
        true_ambient_temp=21.5, 
        true_friction=0.45, 
        eva_pos="living_room", 
        busyness="medium"
    )
    perception.update(raw_observation)
    print(f"Raw Sensor Observation : {raw_observation}")
    print(f"Updated Internal Belief : Temp={belief_state.ambient_temperature_c}°C, Friction={belief_state.surface_friction_coefficient}")
    
    print("\n=== STEP 2: Instruction Interpretation ===")
    raw_command = "Bring me my coffee fast!"
    goal = interpreter.interpret(raw_command, belief_state)
    
    print("\n=== STEP 3: System 1 Proposer Generates Parameters ===")
    proposal_v1 = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.8, 
        tilt_rate_radps=0.2
    )
    print(f"Proposer suggests: v_max = {proposal_v1.max_velocity_mps} m/s")
    
    print("\n=== STEP 4: System 2 SDT & Warrant Gate Validation (Attempt 1) ===")
    warrant_v1 = warrant_gate.validate(proposal_v1, goal, belief_state)
    print(f"SDT Predictions: Nav-Vel = {warrant_v1['predictions']['predicted_nav_velocity']} m/s")
    print(f"Warrant Checks  : {warrant_v1['warrant_checks']}")
    print(f"Gate Verdict    : REJECTED (Failed on: {warrant_v1['failed_criterion']})")
    
    executive.record_trace(raw_command, goal, belief_state, proposal_v1, warrant_v1, raw_observation, committed=False)
    
    print("\n=== STEP 5: Proposer Resamples (Attempt 2) ===")
    proposal_v2 = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.45, 
        tilt_rate_radps=0.08
    )
    
    warrant_v2 = warrant_gate.validate(proposal_v2, goal, belief_state)
    print(f"SDT Predictions: Force = {warrant_v2['predictions']['predicted_force_n']} N, Wave = {warrant_v2['predictions']['predicted_wave_mm']} mm")
    print(f"Warrant Checks  : {warrant_v2['warrant_checks']}")
    print(f"Gate Verdict    : APPROVED")

    #Task2.2 with different velocity
    proposal_v3 = ParameterPacket(
            grasp_pose=(0.45, 0.12, 0.30),
            max_velocity_mps=0.35, 
            tilt_rate_radps=0.08
        )
    warrant_v3 = warrant_gate.validate(proposal_v3, goal, belief_state)
    print("\n=== STEP 5: Proposer with different velocity (Attempt 3) ===")
    print(f"SDT Predictions: Force = {warrant_v3['predictions']['predicted_force_n']} N, Wave = {warrant_v3['predictions']['predicted_wave_mm']} mm")
    print(f"Warrant Checks  : {warrant_v3['warrant_checks']}")
    
    print("\n=== STEP 6: Physical Execution & Epistemic Audit Trail ===")
    if warrant_v2["passed"]:
        final_outcome = executive.execute_action(proposal_v2, goal, warrant_v2)
        executive.record_trace(raw_command, goal, belief_state, proposal_v2, warrant_v2, raw_observation, committed=True, execution_outcome=final_outcome)
        print(f"Execution Outcome: {final_outcome}")

    print(f"\n--- COMPLETE EPISODIC AUDIT TRAIL ({len(executive.audit_trail)} Entries) ---")
    for idx, trace in enumerate(executive.audit_trail, 1):
        status = trace['committed']['action_approved']
        outcome = trace.get('execution_outcome')
        outcome_str = f" | Outcome: {outcome['status']} (Spill: {outcome['spill_occurred']})" if outcome else ""
        print(f"Entry #{idx} | Approved: {status}{outcome_str} | Command: '{trace['asked']['raw_utterance']}'")


    #Task 3.2
    print("Task3.2: Sensing & Perception Update with new ambient temperature and tilt rate")
    raw_observation = sensing.sense(
            true_ambient_temp=28, 
            true_friction=0.45, 
            eva_pos="living_room", 
            busyness="medium"
        )
    perception.update(raw_observation)
    raw_command = "Bring me my coffee fast!"
    goal = interpreter.interpret(raw_command, belief_state)
    print(f"Raw Sensor Observation : {raw_observation}")
    print(f"Updated Internal Belief : Temp={belief_state.ambient_temperature_c}°C, Friction={belief_state.surface_friction_coefficient}")
    proposal_v3_2 = ParameterPacket(
            grasp_pose=(0.45, 0.12, 0.30),
            max_velocity_mps=0.45, 
            tilt_rate_radps=0.15
        )
    warrant_v3_2 = warrant_gate.validate(proposal_v3_2, goal, belief_state)

    print("\n=== STEP 7: Warrant Validation with Updated Perception ===")
    print(f"SDT Predictions: Force = {warrant_v3_2['predictions']['predicted_force_n']} N, Wave = {warrant_v3_2['predictions']['predicted_wave_mm']} mm")
    print(f"Warrant Checks  : {warrant_v3_2['warrant_checks']}")

    #Check for battery
    print("\n=== STEP 8: Battery Check ===")
    if not warrant_v3_2["warrant_checks"]["energetically_viable"]:
        executive.record_trace(
        raw_command, goal, belief_state, proposal_v2, warrant_v2,
        raw_observation, committed=False
    )
        print("[EXECUTIVE] Battery is low; requesting recharge instead of coffee delivery.")
        requested_action = {
            "action": "recharge",
            "reason": "energetically_viable check failed",
        }

=== STEP 1: Sensing & Perception Update ===
Raw Sensor Observation : Observation(timestamp=1793459341.908822, measured_temperature_c=21.55, measured_friction=0.42, eva_position='living_room', room_busyness='medium')
Updated Internal Belief : Temp=21.55°C, Friction=0.42

=== STEP 2: Instruction Interpretation ===

[INTERPRETATION] Analyzing instruction: 'Bring me my coffee fast!' in context of Eva (Position: living_room, Room Busyness: medium)
 -> Derived Goal: Object=blue_ceramic_mug, MaxForce=4.5N, MaxNavVel=0.72m/s, TimeFrame=30.0s

=== STEP 3: System 1 Proposer Generates Parameters ===
Proposer suggests: v_max = 0.8 m/s

=== STEP 4: System 2 SDT & Warrant Gate Validation (Attempt 1) ===
SDT Predictions: Nav-Vel = 0.72 m/s
Warrant Checks  : {'grounded': True, 'feasible': True, 'safe': False, 'monitorable': True, 'repairable': True, 'recordable': True}
Gate Verdict    : REJECTED (Failed on: safe)

=== STEP 5: Proposer Resamples (Attempt 2) ===
SDT Predictions: Force = 4.11 N, Wave = 3

## Questions (0.25 Pt x 4 + 0.5 Pt)

### Task 1: Architectural Mapping & Code Decomposition
* **Question 1.1:** Map the classes and methods implemented in your Python script directly onto the functional blocks of the lecture integration diagram. Explain how data flows between these components during a single cognitive decision cycle.
* **Question 1.2:** Complete the proposer class based on a probabilistic model of your choice (e.g., uniform, gaussian, mixture). Justify your choice.
  

### Task 2: Non-Linear Contact Mechanics and Material Compliance
In the base script, the contact force on the flexible pouch is modeled using a simplified power-law equation. 
* **Question 2.1:** Modify the `SemanticDigitalTwin.predict_properties` method to account for a dual-stage material compliance curve. If the contact depth exceeds $0.02 \, \text{m}$, the stiffness coefficient should increase exponentially by a factor of $1.8$ due to bottoming out against a rigid internal fixture.
* **Question 2.2:** Write a test case where the Proposer generates an aggressive approach velocity ($0.35 \, \text{m/s}$). Calculate the predicted force and verify that the Warrant Gate successfully intercepts and rejects the parameter packet before physical execution.

### Task 3: Advanced Fluid Sloshing and Viscosity Temperature Coupling
Fluids in laboratory automation often exhibit temperature-dependent viscosity changes.
* **Question 3.1:** Enhance the fluid dynamics model so that ambient temperature fluctuations directly alter the effective fluid viscosity using an Arrhenius-type approximation or a linear thermal degradation coefficient ($1\% \, \text{drop per} \, ^\circ\text{C}$ above $20^\circ\text{C}$).
* **Question 3.2:** Simulate an environment where the ambient temperature rises to $28^\circ\text{C}$. How does this change the predicted wave height for a tilt rate of $0.15 \, \text{rad/s}$? Does the Warrant Gate trigger a safety violation?

### Task 4: Expanding the Warrant Gate Checklist
The current Warrant Gate evaluates six core criteria (*grounded, feasible, safe, monitorable, repairable, recordable*). 
* **Question 4.1:** Add a seventh criterion called **`energetically_viable`** to the Warrant Gate dictionary checks. It must evaluate whether the estimated battery drain for the requested trajectory stays below $15\%$ of total capacity.
* **Question 4.2:** Introduce a low-battery state into the `BeliefState` and demonstrate how a failed `energetically_viable` check redirects the system workflow to request a recharge action rather than proceeding with the coffee delivery.

### Task 5: Post-Hoc Epistemic Audit Analysis
A core pillar of trustworthy robotics is the ability to audit *why* a decision was made or rejected.
* **Question 5.1:** Write a post-processing function (`audit_analyzer(audit_trail)`) that parses the recorded episodic log. It should compute and print:
  1. The total number of rejected proposals versus approved executions.
  2. The most frequent warrant criterion responsible for rejection.
  3. The average variance between predicted forces and observed physical outcomes.
* **Question 5.2:** Discuss in your lab report: *Why does maintaining separate records for `asked`, `belief`, `committed`, and `observed` solve the epistemological gap that pure data-driven imitation learning cannot address?*

*All code is directly implemented in the same cell as the exercise*

**Task1.1**:
Perception in this code is implemented as follows: a SensingSystem class senses the environment and return raw readings as an Observation object. This observation object is then used to update the internalBelief class of the robot and ground the raw sensed data. The KnowledgeBase and the InterpretationSystem class form the Interpretation and block. The InterpretationSystem class uses the request and knowledge base to interpret the request and return a specific goal (goalSpecification object) that needs to be achieved. Currently, the proposed parameters represent the action, and they are hardcoded. This proposed action (ParameterPacket object) is then validated by the WarrantGate (decide: warrant gate block) and it uses the semantic digital twin class to predict the sim-to-real epistemic gap, then the WarrantGate uses these results against safety criteria to accept or reject a proposed action. Finally the record_trace inside the Exuction engine records the actions and its tested outcomes by the WarrantGate.

So a single cycle: InterpretationSystem + knowledge base --> goal. SensingSystem --> Observation --> update internal belief Proposed action goes into WarrantGate that uses a semantic digiatl twin to test the validity of the action.

**Task1.2**:
I have used hybrid. I used a gaussian for the pose because the things the robot will interact with are going to be mostly 'within reach' while other things might be on the extremes of reach. On the other hand velocity and tilt were randomized uniformaly while defining limits so that the robot doesn't have a velocity larger than what its hardware can handle, or a non sensical velocity in the negatives.

**Task2.1**:
The contact depth depends on the penetration on the z-axis, which relies on the impact velocity on the z-axis. This then is used to modify the stiffness coeficient.

**Task2.2**:
Results after running the new proposal. Force = 2.87, safe: False

**Task3.1**:The code already uses a linear thermal degradation as specified. I added an implementation for an Arhenius based on Andrade's equation (commented out)

$$ \mu(T) = \mu_{ref} \cdot e^{B \left( \frac{1}{T} - \frac{1}{T_{ref}} \right)} $$

Where:
* **$\mu_{ref}$** is the baseline fluid viscosity at a known temperature (e.g., at $20^\circ\text{C}$)
* **$T_{ref}$** is the baseline temperature in Kelvin ($293.15\text{ K}$)
* **$T$** is the current ambient temperature in Kelvin
* **$B$** is a single fitting constant ($B = E_a / R$) measured in Kelvin

Source [Wiley Online Library](https://onlinelibrary.wiley.com/doi/10.1155/2015/163262).

**Task3.2**: New wave height 6.77 mm. The Warrant Gate triggered a safety violation, safe = False (See step seven in the output)

**Task4.1**: Added energy estimate calculation based on the distance the robot will travel according to the trajectory and included the check in the warrant gate.

**Task4.2**: Added the low battery boolean variable to the belief state and included a requested action to charging when the battery is low.




In [2]:
#Task5.1

from collections import Counter
from statistics import mean, pvariance


def audit_analyzer(audit_trail:List[Dict[str, Any]]):
    rejected = 0
    approved = 0
    rejection_criteria = Counter()
    force_errors = []

    for record in audit_trail:
        committed = record.get("committed", {})
        was_approved = committed.get("action_approved")
        warrant = committed.get("warrant_verdict", {})

        if was_approved is True:
            approved += 1
        elif was_approved is False:
            rejected += 1
            criterion = warrant.get("failed_criterion")
            if criterion:
                rejection_criteria[criterion] += 1

        predictions = warrant.get("predictions", {})
        outcome = record.get("execution_outcome") or {}
        predicted_force = predictions.get("predicted_force_n")
        actual_force = outcome.get("actual_force_n")

        if isinstance(predicted_force, (int, float)) and isinstance(actual_force, (int, float)):
            force_errors.append(actual_force - predicted_force)

    print(f"Rejected proposals: {rejected}")
    print(f"Approved executions: {approved}")

    if rejection_criteria:
        criterion, count = rejection_criteria.most_common(1)[0]
        print(f"Most frequent rejection criterion: {criterion} ({count} times)")
    else:
        print("Most frequent rejection criterion: none recorded")

    if force_errors:
        average_absolute_error = mean(abs(error) for error in force_errors)
        error_variance = pvariance(force_errors)
        print(f"Average absolute force error: {average_absolute_error:.2f} N")
        print(f"Population variance of force errors: {error_variance:.4f} N²")
    else:
        average_absolute_error = None
        error_variance = None
        print("Force error statistics: no recorded executions with predicted and actual force")

    return {
        "rejected_proposals": rejected,
        "approved_executions": approved,
        "rejection_criteria": dict(rejection_criteria),
        "average_absolute_force_error_n": average_absolute_error,
        "force_error_variance_n2": error_variance,
    }
print("\n=== STEP 9: Audit Trail Analysis ===")
audit_results = audit_analyzer(executive.audit_trail)
print(f"Audit Results: {audit_results}")


=== STEP 9: Audit Trail Analysis ===
Rejected proposals: 1
Approved executions: 0
Most frequent rejection criterion: safe (1 times)
Force error statistics: no recorded executions with predicted and actual force
Audit Results: {'rejected_proposals': 1, 'approved_executions': 0, 'rejection_criteria': {'safe': 1}, 'average_absolute_force_error_n': None, 'force_error_variance_n2': None}


**Task5.2**: In imitation learning, what the robot learns is a black box, here we can see the exact decision making process. We can seee for example what was asked from the robot and how is that separated from what the robot observed. Imitation learning also suffers when the real world doesn't agree with simulation, here we have the observed ground truth and what it believes, this allows us to track and diagnose errors caused by false set of beliefs that were not grounded with the real data, unlike in imitation learning where we can't peek into that. This closes the epistemological gap between reality and what is believed. Finally the committed and warrant verdict provide safety net and boundaries that data driven models can't put. A simulation might learn to do things that are not safe and should have been filtered out. It also provides a transperent way to look into justifications the robot made that we can't infer from black box imitation simulation.

Another important aspect is the engineering tinkering you could do with this approach, you can peak right into the robot's 'mind'. For example you can run offline simulations where you change only the belief layer to see if the robot's planning would have succeeded. For example: "If the robot's belief map had correctly registered the room's busyness, would its proposed velocity have cleared the safety warrant?" and so on, which you can't make with data based learning models.

## Part 3: Robot Anatomy (0.5 Pt)

**Title:** Links Vs Joints, Actuators Vs Sensors, Effectors Vs Bases Vs Torsos Vs Heads, Wheels Vs Legs Vs Grippers, Proprioception Vs Exteroception Vs Interoception

![Alternative Text](robot-body.jpg)

### Questions

Identify the key functional parts of a robot based on the above picture and briefly indicate how they support the execution of function in the above architecture.

---

*Answer Part3*


***Links vs Joints***: Links are rigid structural bodies of the robot, while the joints, like the shoulder, elbow and hand joints, connect the links and provide mobility to them. The robot's shoulder, elbow and hand joints form a chain that facilitates the movements of the robots's end effector, the gripper, allowing it to grap the cup of coffee or open drawers. 

***Actuators vs Sensors***: Actuators convert electrical signals into physical motion, the motors shown at the base in the picture are an example of an actuator, they control the movement of  the wheels to make the robot navigate its environement and reach its sesired destination. On the other hand, sensors collect data from the robot's environment and provide this data for the robot to act upon, they provide the perception the robot needs.

***Effectors vs Bases vs Torsos vs Heads***: The gripper acts as an end effector for the arm. The robot uses it to grap onto the coffee mug and open drawers. The photo shows also the base of the robot which provides the navigation capibality.The torso supports the rest of the robot, while the head has the primary vision sensors(cameras) at an elevated point.


***Wheels vs. Legs***: The choice for using wheels makes sense for a home assistant because it will not face rough terrain (like mountains), so it's mobility in the house will not be limited by its wheels, allowing it to navigate through various rooms with agility.

***Proprioception vs. Exteroception vs. Interoception***: For perception, the photo shows laser sensors that are used to measure distances to objects and people to avoid collisions, and also cameras that capture visual data from the surrounding and enable the robot to detect objects and identify where the mug is. However these perception sensors are not just exteroception sensors, the robot is also equipped with proprioception sensors that allows it to measure its own physical state like the sensors in the gripper to measure the pressure applied to the mug such that the robot can adjust the pressure exerted depending on the object. Also sensors that measure the tilt of the mug while carriying it. Morever, the robot is equipped with interoception sensors that percieve the internal state of the robot, like its battery levels to signal a recharge action when needed.




# Submission guidelines

- Deadline: 03.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (Create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put all the answers in the notebook. Any other media related to your answer should be inserted in the notebook as well. The notebook and all related media should then be put into a folder called **Assignment1**. 